In [1]:
import csv
from datetime import datetime

TIPOS_VALIDOS = {"credito", "debito"}
CAMINHO_ARQUIVO = "transacoes.csv"
LIMITE_SUSPEITO = 10000.00

In [2]:
import csv

transacoes = [
    # id, data, cliente_id, tipo, valor, descricao, categoria

    # ---------- REGISTROS VÁLIDOS (15) — distribuídos em jan/fev/mar 2026 ----------
    [1, "2026-01-05", "CLI001", "credito", "3500.00", "Salário Janeiro", "salario"],
    [2, "2026-01-10", "CLI002", "debito", "180.50", "Supermercado", "compra"],
    [3, "2026-01-15", "CLI003", "debito", "250.00", "Farmácia", "compra"],
    [4, "2026-01-22", "CLI001", "debito", "120.00", "Internet", "conta"],
    [5, "2026-01-28", "CLI004", "credito", "12000.00", "Bônus", "transferencia"],
    [6, "2026-02-03", "CLI002", "credito", "4200.00", "Salário Fevereiro", "salario"],
    [7, "2026-02-08", "CLI005", "debito", "350.00", "Conta de Luz", "conta"],
    [8, "2026-02-12", "CLI001", "debito", "99.90", "Streaming", "assinatura"],
    [9, "2026-02-20", "CLI003", "credito", "15000.00", "Transferência", "transferencia"],
    [10, "2026-02-25", "CLI004", "debito", "890.00", "Viagem", "compra"],
    [11, "2026-03-01", "CLI001", "credito", "3500.00", "Salário Março", "salario"],
    [12, "2026-03-07", "CLI002", "debito", "600.00", "Aluguel", "conta"],
    [13, "2026-03-15", "CLI003", "debito", "230.00", "Mercado", "compra"],
    [14, "2026-03-22", "CLI005", "credito", "1800.00", "Freelancer", "transferencia"],
    [15, "2026-03-29", "CLI004", "debito", "420.00", "Academia", "assinatura"],

    # ---------- REGISTROS INVÁLIDOS (5) — um para cada regra de validação ----------
    ["", "2026-03-30", "CLI002", "debito", "300.00", "ID vazio", "compra"],           # id vazio
    [17, "2026-15-10", "CLI002", "credito", "500.00", "Data inválida", "salario"],    # mês 15 não existe
    [18, "2026-03-20", "", "debito", "200.00", "Cliente vazio", "compra"],            # cliente_id vazio
    [19, "2026-03-25", "CLI001", "pix", "150.00", "Tipo inválido", "conta"],          # tipo fora do domínio
    [20, "2026-03-28", "CLI004", "credito", "abc", "Valor inválido", "salario"],      # valor não numérico
]

colunas = ["id", "data", "cliente_id", "tipo", "valor", "descricao", "categoria"]

with open("transacoes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(colunas)
    writer.writerows(transacoes)

print(f"Arquivo 'transacoes.csv' criado com {len(transacoes)} registros.")

Arquivo 'transacoes.csv' criado com 20 registros.


In [3]:
import csv

def ler_transacoes(caminho):
    try:
        with open(caminho, newline="", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            transacoes = list(leitor)
        return transacoes
    except FileNotFoundError:
        print(f"[ERRO] Arquivo '{caminho}' não encontrado.")
        return None

#Aqui eu normalizo os dados


In [4]:
def normalizar_id(valor_bruto):
    try:
        return int(valor_bruto.strip())
    except (ValueError, AttributeError):
        return None


def normalizar_cliente_id(valor_bruto):
    cliente_id = valor_bruto.strip()
    if not cliente_id:
        return None
    return cliente_id


def normalizar_data(valor_bruto):
    try:
        return datetime.strptime(valor_bruto.strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None


def normalizar_tipo(valor_bruto):
    tipo = valor_bruto.strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None
    return tipo


def normalizar_valor(valor_bruto):
    try:
        valor = float(valor_bruto.strip())
    except (ValueError, AttributeError):
        return None
    if valor <= 0:
        return None
    return valor


def normalizar_texto(valor_bruto):
    return valor_bruto.strip().lower()

#Aqui, após normalizar os dados eu valido cada linha

In [5]:
def validar_transacao(linha):
    id_transacao = normalizar_id(linha.get("id", ""))
    if id_transacao is None:
        return None

    cliente_id = normalizar_cliente_id(linha.get("cliente_id", ""))
    if cliente_id is None:
        return None

    data = normalizar_data(linha.get("data", ""))
    if data is None:
        return None

    tipo = normalizar_tipo(linha.get("tipo", ""))
    if tipo is None:
        return None

    valor = normalizar_valor(linha.get("valor", ""))
    if valor is None:
        return None

    return {
        "id": id_transacao,
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": normalizar_texto(linha.get("descricao", "")),
        "categoria": normalizar_texto(linha.get("categoria", "")),
    }

In [6]:
transacoes_brutas = ler_transacoes(CAMINHO_ARQUIVO)

transacoes_validas = []  # essa lista aqui é a nossa "tabela" pronta pra gerar métrica
total_invalidas = 0

if transacoes_brutas is not None:
    for linha in transacoes_brutas:
        resultado = validar_transacao(linha)
        if resultado is not None:
            transacoes_validas.append(resultado)
        else:
            total_invalidas += 1

    print("Total de linhas lidas:", len(transacoes_brutas))
    print("Linhas válidas:", len(transacoes_validas))
    print("Linhas inválidas:", total_invalidas)

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


#Aqui eu confiro as primeiras transações validas pra ver se os dados já estão tipados certo

In [7]:
for t in transacoes_validas[:5]:  # só as 5 primeiras pra conferir
    print(t["id"], t["data"].strftime("%Y-%m-%d"), t["cliente_id"], t["tipo"], t["valor"])

1 2026-01-05 CLI001 credito 3500.0
2 2026-01-10 CLI002 debito 180.5
3 2026-01-15 CLI003 debito 250.0
4 2026-01-22 CLI001 debito 120.0
5 2026-01-28 CLI004 credito 12000.0


#Aqui eu calculo as métricas, pegando o período analisado, o resumo por mês e as transações suspeitas

In [8]:
def gerar_relatorio(transacoes_validas):
    datas = [t["data"] for t in transacoes_validas]
    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)
    dias_entre = (data_mais_recente - data_mais_antiga).days

    resumo_mensal = {}
    for t in transacoes_validas:
        mes = t["data"].strftime("%Y-%m")
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "valores": [],  # so uso pra calcular media/maior/menor, apago depois
            }
        m = resumo_mensal[mes]
        m["quantidade"] += 1
        m["valores"].append(t["valor"])
        if t["tipo"] == "credito":
            m["total_credito"] += t["valor"]
        else:
            m["total_debito"] += t["valor"]

    # fecho cada mes com saldo, media, maior e menor valor
    for mes, m in resumo_mensal.items():
        m["total_credito"] = round(m["total_credito"], 2)
        m["total_debito"] = round(m["total_debito"], 2)
        m["saldo"] = round(m["total_credito"] - m["total_debito"], 2)
        m["media"] = round(sum(m["valores"]) / len(m["valores"]), 2)
        m["maior_valor"] = round(max(m["valores"]), 2)
        m["menor_valor"] = round(min(m["valores"]), 2)
        del m["valores"]

    transacoes_suspeitas = []
    for t in transacoes_validas:
        if t["valor"] > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": t["valor"],
            })

    return {
        "data_mais_antiga": data_mais_antiga.strftime("%Y-%m-%d"),
        "data_mais_recente": data_mais_recente.strftime("%Y-%m-%d"),
        "dias_entre": dias_entre,
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": transacoes_suspeitas,
    }

#Aqui eu formato os valores em reais e monto o relatório formatado pra aparecer no terminal

In [9]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio, total_lidas, total_validas, total_invalidas):
    print("===== RELATÓRIO MENSAL =====")
    print()
    print(f"Período analisado: {relatorio['data_mais_antiga']} → {relatorio['data_mais_recente']} ({relatorio['dias_entre']} dias)")
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Transações válidas: {total_validas}")
    print(f"Transações inválidas: {total_invalidas}")
    print()

    for mes, m in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {m['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(m['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(m['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(m['saldo'])}")
        print(f"  Média:         {formatar_moeda(m['media'])}")
        print(f"  Maior valor:   {formatar_moeda(m['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(m['menor_valor'])}")
        print()

    print("===== TRANSAÇÕES SUSPEITAS =====")
    if relatorio["transacoes_suspeitas"]:
        for s in relatorio["transacoes_suspeitas"]:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")

#Aqui eu salvo o relatório em um arquivo JSON

In [10]:
import json

def salvar_json(relatorio, total_lidas, total_validas, total_invalidas, caminho="relatorio.json"):
    saida = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_lidas": total_lidas,
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "periodo_analisado": {
            "data_mais_antiga": relatorio["data_mais_antiga"],
            "data_mais_recente": relatorio["data_mais_recente"],
            "dias_entre": relatorio["dias_entre"],
        },
        "resumo_mensal": relatorio["resumo_mensal"],
        "transacoes_suspeitas": relatorio["transacoes_suspeitas"],
    }
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(saida, arquivo, ensure_ascii=False, indent=2)
    print(f"Relatório salvo em '{caminho}'.")

#Aqui eu chamo todas as funções juntas pra rodar o fluxo completo

In [11]:
if transacoes_validas:
    relatorio = gerar_relatorio(transacoes_validas)
    exibir_relatorio(relatorio, len(transacoes_brutas), len(transacoes_validas), total_invalidas)
    salvar_json(relatorio, len(transacoes_brutas), len(transacoes_validas), total_invalidas)
else:
    print("Nenhuma transação válida encontrada, relatório não gerado.")

===== RELATÓRIO MENSAL =====

Período analisado: 2026-01-05 → 2026-03-29 (83 dias)
Total de linhas lidas: 20
Transações válidas: 15
Transações inválidas: 5

Mês: 2026-01
  Transações: 5
  Total crédito: R$ 15.500,00
  Total débito:  R$ 550,50
  Saldo:         R$ 14.949,50
  Média:         R$ 3.210,10
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 120,00

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 19.200,00
  Total débito:  R$ 1.339,90
  Saldo:         R$ 17.860,10
  Média:         R$ 4.107,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 5.300,00
  Total débito:  R$ 1.250,00
  Saldo:         R$ 4.050,00
  Média:         R$ 1.310,00
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 230,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 5 | Cliente: CLI004 | Data: 2026-01-28 | Valor: R$ 12.000,00
ID: 9 | Cliente: CLI003 | Data: 2026-02-20 | Valor: R$ 15.000,00
Relatório salvo em 'relatorio.json'.
